# Ollama Remote Connection And Model Access Test

This notebook only checks whether the Ollama server at `10.0.0.201:8000` is reachable and whether each listed generation and embedding model can be used successfully.


In [3]:
from __future__ import annotations

import json
import os
import socket
import sys
import urllib.error
import urllib.request
from pathlib import Path
from pprint import pprint
from urllib.parse import urlparse

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

DEFAULT_OLLAMA_ENDPOINT = "http://10.0.0.201:8000"
OLLAMA_ENDPOINT = os.environ.get("OLLAMA_ENDPOINT", DEFAULT_OLLAMA_ENDPOINT).rstrip("/")
parsed_endpoint = urlparse(OLLAMA_ENDPOINT)
resolved_host = "unresolved"
try:
    resolved_host = socket.gethostbyname(parsed_endpoint.hostname or "")
except OSError as exc:
    resolved_host = f"resolution_failed: {exc}"

generation_models = [
    "gemma4",
    "qwen3.5:9b",
    "medgemma1.5",
]

embedding_models = [
    "qwen3-embedding:0.6b",
    "embeddinggemma:latest",
    "all-minilm:latest",
]

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())
print("OLLAMA_ENDPOINT:", OLLAMA_ENDPOINT)
print("Resolved host:", resolved_host)
print("generation_models:")
pprint(generation_models)
print("embedding_models:")
pprint(embedding_models)


Python executable: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes/venv/bin/python
Working directory: /Users/oluwatosinoso/Library/CloudStorage/OneDrive-hull.ac.uk/argumentation_schemes
OLLAMA_ENDPOINT: http://10.0.0.201:8000
Resolved host: 10.0.0.201
generation_models:
['gemma4', 'qwen3.5:9b', 'medgemma1.5']
embedding_models:
['qwen3-embedding:0.6b', 'embeddinggemma:latest', 'all-minilm:latest']


In [4]:
def ollama_post(path: str, payload: dict) -> dict:
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_ENDPOINT}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.loads(response.read().decode("utf-8"))


def ollama_get(path: str) -> dict:
    with urllib.request.urlopen(f"{OLLAMA_ENDPOINT}{path}", timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def list_available_models() -> set[str]:
    payload = ollama_get("/api/tags")
    return {item["name"] for item in payload.get("models", [])}


def check_tcp_connectivity() -> str:
    parsed = urlparse(OLLAMA_ENDPOINT)
    host = parsed.hostname or ""
    port = parsed.port or (443 if parsed.scheme == "https" else 80)
    try:
        with socket.create_connection((host, port), timeout=5):
            return f"tcp_ok host={host} port={port}"
    except OSError as exc:
        return f"tcp_failed host={host} port={port} error={type(exc).__name__}: {exc}"


def test_generation_model(model: str) -> dict:
    try:
        payload = ollama_post(
            "/api/generate",
            {
                "model": model,
                "prompt": "Reply with the single word OK.",
                "stream": False,
                "options": {"num_predict": 8},
            },
        )
        return {
            "model": model,
            "kind": "generation",
            "ok": True,
            "detail": (payload.get("response") or "").strip(),
        }
    except Exception as exc:
        return {
            "model": model,
            "kind": "generation",
            "ok": False,
            "detail": f"{type(exc).__name__}: {exc}",
        }


def test_embedding_model(model: str) -> dict:
    try:
        payload = ollama_post(
            "/api/embed",
            {
                "model": model,
                "input": "Embedding connectivity check.",
            },
        )
        embeddings = payload.get("embeddings") or []
        length = len(embeddings[0]) if embeddings and embeddings[0] else 0
        return {
            "model": model,
            "kind": "embedding",
            "ok": length > 0,
            "detail": f"embedding_length={length}",
        }
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            return {
                "model": model,
                "kind": "embedding",
                "ok": False,
                "detail": f"HTTPError: {exc}",
            }
        try:
            payload = ollama_post(
                "/api/embeddings",
                {
                    "model": model,
                    "prompt": "Embedding connectivity check.",
                },
            )
            embedding = payload.get("embedding") or []
            length = len(embedding)
            return {
                "model": model,
                "kind": "embedding",
                "ok": length > 0,
                "detail": f"embedding_length={length}",
            }
        except Exception as fallback_exc:
            return {
                "model": model,
                "kind": "embedding",
                "ok": False,
                "detail": f"{type(fallback_exc).__name__}: {fallback_exc}",
            }
    except Exception as exc:
        return {
            "model": model,
            "kind": "embedding",
            "ok": False,
            "detail": f"{type(exc).__name__}: {exc}",
        }


In [5]:
try:
    available_models = list_available_models()
    print(f"Server reachable. {len(available_models)} model(s) reported by /api/tags.")

    missing_generation = [model for model in generation_models if model not in available_models]
    missing_embedding = [model for model in embedding_models if model not in available_models]

    print("Missing generation models:", missing_generation or "None")
    print("Missing embedding models:", missing_embedding or "None")
except Exception as exc:
    print("Server check failed:", f"{type(exc).__name__}: {exc}")
    print("TCP diagnostic:", check_tcp_connectivity())
    print("Tip: if the host works in your shell but not in VS Code, compare the selected notebook kernel and whether VS Code was launched from the same shell session.")
    raise


Server reachable. 7 model(s) reported by /api/tags.
Missing generation models: ['gemma4', 'medgemma1.5']
Missing embedding models: None


In [6]:
results = []

for model in generation_models:
    results.append(test_generation_model(model))

for model in embedding_models:
    results.append(test_embedding_model(model))

results


[{'model': 'gemma4', 'kind': 'generation', 'ok': True, 'detail': 'OK'},
 {'model': 'qwen3.5:9b', 'kind': 'generation', 'ok': True, 'detail': ''},
 {'model': 'medgemma1.5',
  'kind': 'generation',
  'ok': False,
  'detail': 'HTTPError: HTTP Error 404: Not Found'},
 {'model': 'qwen3-embedding:0.6b',
  'kind': 'embedding',
  'ok': True,
  'detail': 'embedding_length=1024'},
 {'model': 'embeddinggemma:latest',
  'kind': 'embedding',
  'ok': True,
  'detail': 'embedding_length=768'},
 {'model': 'all-minilm:latest',
  'kind': 'embedding',
  'ok': True,
  'detail': 'embedding_length=384'}]

In [7]:
try:
    import pandas as pd

    df = pd.DataFrame(results)
    display(df)
    summary = df.groupby(["kind", "ok"]).size().rename("count").reset_index()
    display(summary)
except ModuleNotFoundError:
    pprint(results)


,model,kind,ok,detail
0,gemma4,generation,True,OK
1,qwen3.5:9b,generation,True,
2,medgemma1.5,generation,False,HTTPError: HTTP Error 404: Not Found
3,qwen3-embedding:0.6b,embedding,True,embedding_length=1024
4,embeddinggemma:latest,embedding,True,embedding_length=768
5,all-minilm:latest,embedding,True,embedding_length=384


,kind,ok,count
0,embedding,True,3
1,generation,False,1
2,generation,True,2
